# Validation of the ERMM-Chart designer

This notebook contains the analyses reported in Section 7 of the article
*Interactive design of a robust EWMA Max-M control chart linking product
specifications to finish-mill operation in cement manufacturing*:

1. agreement of the Python reweighted Fast-MCD estimator with `robustbase::covMcd` (R);
2. the constants of the Max-M statistic;
3. control limits of the case design (N = 100, p = 5, ARL0 = 370) against Table 10 of the article;
4. out-of-control ARLs against Table 11;
5. the specification-driven design rule (Eqs. 8 and 9) for the LOI limit;
6. the synthetic demonstration data and the full workflow of the app (Figs. 14 to 16);
7. computing times.

The 10,000-replication calibrations take about 2.5 min each for the robust
charts. They were run with `notebooks/calib_cache.py`, which calls the same
function `ermm.design.calibrate` with the seeds stored in the cache; cell 3
recomputes one design live and checks that it is identical.

In [1]:
import sys, time, json, subprocess, tempfile, platform
from pathlib import Path
import numpy as np, pandas as pd, scipy
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import ermm
from ermm.estimators import reweighted_fast_mcd, classical_estimate
from ermm.charts import MU_M, SIGMA_M, ermm_path, L_from_ucl
from ermm.design import calibrate, evaluate_arl, uniform_shift, spec_driven_shifts, Shift
from ermm.diagnosis import (signal_table, first_signal, component_summary,
                            characteristic_deviations, rank_actions, property_comparison)
sys.path.insert(0, str(ROOT / "notebooks"))
import calib_cache
pd.set_option("display.width", 160)
print("ermm", ermm.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__,
      "| pandas", pd.__version__, "| python", platform.python_version())
print("R:", subprocess.run(["Rscript", "-e", "cat(R.version.string, '| robustbase', as.character(packageVersion('robustbase')))"],
                           capture_output=True, text=True, check=True).stdout)

ermm 1.0.0 | numpy 2.5.4 | scipy 1.18.1 | pandas 3.0.6 | python 3.12.3
R: R version 4.3.3 (2024-02-29) | robustbase 0.99.2


## 1 Estimator: Python versus robustbase
The test data sets in `tests/data` are re-estimated in R now and compared with the Python implementation.

In [2]:
DATA = ROOT / "tests" / "data"
tmp = Path(tempfile.mkdtemp())
rcode = '''library(robustbase)
for (k in c(0, 1, 3, 4)) {
  x <- as.matrix(read.csv(file.path("%s", sprintf("x%%d.csv", k)), header = FALSE))
  set.seed(123); f <- covMcd(x, alpha = 0.75, nsamp = 500)
  write.table(f$center, file.path("%s", sprintf("c%%d.txt", k)), row.names = FALSE, col.names = FALSE)
  write.table(f$cov, file.path("%s", sprintf("v%%d.txt", k)), row.names = FALSE, col.names = FALSE)
  write.table(f$raw.weights, file.path("%s", sprintf("w%%d.txt", k)), row.names = FALSE, col.names = FALSE)
}''' % (DATA, tmp, tmp, tmp)
subprocess.run(["Rscript", "-e", rcode], check=True)
rows = []
for k in [0, 1, 3, 4]:
    X = np.loadtxt(DATA / f"x{k}.csv", delimiter=",")
    N, p = X.shape
    est = reweighted_fast_mcd(X, 0.75, 500, 0.975, np.random.default_rng(7))
    rc, rv, rw = (np.loadtxt(tmp / f"c{k}.txt"), np.loadtxt(tmp / f"v{k}.txt").reshape(p, p),
                  np.loadtxt(tmp / f"w{k}.txt"))
    rows.append(dict(N=N, p=p, h=est.h, excluded=int((est.weights == 0).sum()),
                     max_abs_diff_centre=np.abs(est.center - rc).max(),
                     max_abs_diff_cov=np.abs(est.cov - rv).max(),
                     same_weights=bool(np.array_equal(est.weights, rw))))
res1 = pd.DataFrame(rows); print(res1.to_string(index=False))
assert res1.same_weights.all() and res1.max_abs_diff_cov.max() < 1e-12

   N  p   h  excluded  max_abs_diff_centre  max_abs_diff_cov  same_weights
 100  5  76         8         4.996004e-16      7.771561e-16          True
 100  5  76        12         4.718448e-16      4.662937e-15          True
  60  2  45         2         2.498002e-16      4.662937e-15          True
1000  3 751        60         5.551115e-17      3.774758e-15          True


In [3]:
rng = np.random.default_rng(1)
Xs = [rng.standard_normal((100, 5)) for _ in range(100)]
t0 = time.perf_counter()
for X in Xs: reweighted_fast_mcd(X, 0.75, 500, 0.975, rng)
t_mcd = (time.perf_counter() - t0) / len(Xs)
print(f"Fast-MCD (N = 100, p = 5, 500 starts): {1000 * t_mcd:.1f} ms per estimate")

Fast-MCD (N = 100, p = 5, 500 starts): 11.7 ms per estimate


## 2 Constants of the Max-M statistic

In [4]:
z = np.random.default_rng(0).standard_normal((2, 4_000_000)); m = np.abs(z).max(axis=0)
print(f"mu_M    = {MU_M:.6f}  (2/sqrt(pi) = {2/np.sqrt(np.pi):.6f}; Monte Carlo {m.mean():.4f})")
print(f"sigma_M = {SIGMA_M:.6f}  (Monte Carlo {m.std():.4f})")

mu_M    = 1.128379  (2/sqrt(pi) = 1.128379; Monte Carlo 1.1279)
sigma_M = 0.602810  (Monte Carlo 0.6029)


## 3 Control limits of the case design
Algorithm 1 with 10,000 replications, horizon 10,000, seed 2026. The standard
error of a calibrated limit is SE(ARL)/slope, where SE(ARL) = SDRL/sqrt(10,000)
and the slope dARL/dlimit is estimated by a local linear fit of the ARL grid.

In [5]:
t0 = time.perf_counter()
live = calib_cache.run_or_load("maxm", "classical", 0.1, recompute=True)
t_live = time.perf_counter() - t0
cached = calib_cache.run_or_load("maxm", "classical", 0.1)
print(f"live recomputation: limit {live['limit']:.6f} in {t_live:.0f} s; cached {cached['limit']:.6f}")
assert live["limit"] == cached["limit"] and live["arl_grid"] == cached["arl_grid"]

live recomputation: limit 1.554557 in 22 s; cached 1.554557


In [6]:
paper = {("maxm","classical",0.1):1.5511, ("maxm","fastmcd",0.1):1.5737, ("maxm","classical",0.3):2.0479,
         ("maxm","fastmcd",0.3):2.0870, ("maxm","classical",0.5):2.4470, ("maxm","fastmcd",0.5):2.4974,
         ("maxm","classical",0.7):2.8029, ("maxm","fastmcd",0.7):2.8649, ("maxm","classical",1.0):3.354,
         ("maxm","fastmcd",1.0):3.427, ("mewma","classical",0.1):19.730, ("mewma","fastmcd",0.1):20.493}
name = {("maxm","classical"):"EMM", ("maxm","fastmcd"):"ERMM", ("mewma","classical"):"MEWMA, classical",
        ("mewma","fastmcd"):"MEWMA, Fast-MCD"}
rows = []
for (chart, method, lam), lit in paper.items():
    r = calib_cache.run_or_load(chart, method, lam)
    g, a, s = map(np.asarray, (r["grid"], r["arl_grid"], r["sdrl_grid"]))
    k = int(np.argmin(np.abs(g - r["limit"]))); lo, hi = max(k - 3, 0), min(k + 4, g.size)
    slope = np.polyfit(g[lo:hi], a[lo:hi], 1)[0]
    se_lim = (s[k] / np.sqrt(r["n_rep"])) / slope
    arl_at_lit = np.exp(np.interp(lit, g, np.log(a))) if g[0] <= lit <= g[-1] else np.nan
    rows.append(dict(chart=name[(chart, method)], lam=lam, article=lit, tool=r["limit"],
                     difference=r["limit"] - lit, se_limit=se_lim, z=(r["limit"] - lit) / se_lim,
                     tool_ARL_at_article_limit=arl_at_lit, SDRL0=s[k], minutes=r["seconds"] / 60))
tab17 = pd.DataFrame(rows)
print(tab17.round({"tool":4,"difference":4,"se_limit":4,"z":2,"tool_ARL_at_article_limit":1,"SDRL0":0,"minutes":2}).to_string(index=False))
mm = tab17[~tab17.chart.str.startswith("MEWMA")]
print(f"Max-M charts: |difference| <= {mm.difference.abs().max():.4f} "
      f"({100*(mm.difference.abs()/mm.article).max():.2f}% of the limit); |z| <= {mm.z.abs().max():.2f}")

           chart  lam  article    tool  difference  se_limit     z  tool_ARL_at_article_limit  SDRL0  minutes
             EMM  0.1   1.5511  1.5546      0.0035    0.0012  3.00                      357.2  453.0     0.29
            ERMM  0.1   1.5737  1.5746      0.0009    0.0013  0.68                      366.8  540.0     2.33
             EMM  0.3   2.0479  2.0510      0.0031    0.0019  1.62                      361.8  458.0     0.25
            ERMM  0.3   2.0870  2.0906      0.0036    0.0022  1.61                      361.1  522.0     2.30
             EMM  0.5   2.4470  2.4449     -0.0021    0.0027 -0.78                      373.3  446.0     0.24
            ERMM  0.5   2.4974  2.5016      0.0042    0.0030  1.43                      363.6  505.0     2.34
             EMM  0.7   2.8029  2.8054      0.0025    0.0031  0.79                      366.3  435.0     0.25
            ERMM  0.7   2.8649  2.8694      0.0045    0.0035  1.31                      363.4  483.0     2.30
          

## 4 Out-of-control ARL against Table 11
Variability shifts do not depend on the correlation matrix and are compared
directly. A mean shift of all characteristics by a standard deviations has the
squared Mahalanobis length a^2 1'R^-1 1, so its ARL depends on the correlation
matrix R used in the article, which was the plant's Phase I estimate.

In [7]:
R5 = np.array([[1,-.23,-.08,.29,.23],[-.23,1,.41,.08,-.42],[-.08,.41,1,.13,-.36],
               [.29,.08,.13,1,-.17],[.23,-.42,-.36,-.17,1.]])        # Table 5
lim = {k: calib_cache.run_or_load(*k)["limit"] for k in [("maxm","classical",0.1),("maxm","fastmcd",0.1),
                                                        ("mewma","classical",0.1),("mewma","fastmcd",0.1)]}
charts = [dict(name=name[(c, m)], chart=c, method=m, lam=l, limit=v) for (c, m, l), v in lim.items()]
shifts = [uniform_shift(5, 0, 1.4), uniform_shift(5, 0, 2.0)]
t0 = time.perf_counter()
oc = pd.DataFrame(evaluate_arl(charts, shifts, R5, 100, 2000, 10000, 99, 0.75, 500, 0.975))
t11 = {("EMM","a=0, b=1.4"):32.6, ("ERMM","a=0, b=1.4"):32.4, ("MEWMA, classical","a=0, b=1.4"):87.4,
       ("MEWMA, Fast-MCD","a=0, b=1.4"):91.2, ("EMM","a=0, b=2"):8.61, ("ERMM","a=0, b=2"):8.64,
       ("MEWMA, classical","a=0, b=2"):33.5, ("MEWMA, Fast-MCD","a=0, b=2"):34.1}
oc["article"] = [t11[(c, s)] for c, s in zip(oc.chart, oc["shift"])]
oc["z"] = (oc.ARL - oc.article) / oc.SE
print(oc[["chart","shift","ARL","SE","SDRL","article","z"]].round(2).to_string(index=False))
print(f"({time.perf_counter()-t0:.0f} s)")
print("1' R^-1 1 for Table 5:", round(np.ones(5) @ np.linalg.solve(R5, np.ones(5)), 2))

           chart      shift   ARL   SE  SDRL  article     z
             EMM a=0, b=1.4 33.12 0.70 31.33    32.60  0.74
             EMM   a=0, b=2  8.67 0.13  5.75     8.61  0.48
            ERMM a=0, b=1.4 31.19 0.75 33.39    32.40 -1.62
            ERMM   a=0, b=2  8.65 0.13  6.04     8.64  0.05
MEWMA, classical a=0, b=1.4 90.87 2.07 92.47    87.40  1.68
MEWMA, classical   a=0, b=2 32.88 0.64 28.63    33.50 -0.97
 MEWMA, Fast-MCD a=0, b=1.4 90.24 2.08 93.03    91.20 -0.46
 MEWMA, Fast-MCD   a=0, b=2 33.34 0.66 29.64    34.10 -1.15
(29 s)
1' R^-1 1 for Table 5: 6.13


## 5 Specification-driven design: LOI limit of SNI 2049 with the plant estimates of Table 7

In [8]:
from scipy import stats
m_loi, s_loi, usl = 4.259, 0.378, 5.0
d = (usl - m_loi) / s_loi
print(f"margin d = {d:.3f} robust SD, C_pk = d/3 = {d/3:.2f}, nonconforming under normality = {stats.norm.sf(d):.2%}")
for pi in (0.05, 0.01):
    r = spec_driven_shifts(["LOI"], np.array([m_loi]), np.array([s_loi]), np.array([np.nan]), np.array([usl]), pi)[0]
    print(f"pi = {pi:.0%}: critical mean shift a = {r['critical_mean_shift_sd']:+.3f} SD, variance factor b = {r['critical_var_factor']:.3f}")

margin d = 1.960 robust SD, C_pk = d/3 = 0.65, nonconforming under normality = 2.50%
pi = 5%: critical mean shift a = +0.315 SD, variance factor b = 1.420
pi = 1%: critical mean shift a = -0.366 SD, variance factor b = 0.710


## 6 Synthetic demonstration data and the app workflow
The data are regenerated with `data/make_synthetic_data.py` and checked against
the published parameters; the workflow then uses exactly the settings of the
app (seed 2026 for Fast-MCD, seed 2027 and 500 replications for the ARL study).

In [9]:
sys.path.insert(0, str(ROOT / "data")); import make_synthetic_data as msd
before = (ROOT / "data" / "opc_synthetic.csv").read_bytes()
msd.main()
assert (ROOT / "data" / "opc_synthetic.csv").read_bytes() == before, "synthetic data not reproducible"
df = pd.read_csv(ROOT / "data" / "opc_synthetic.csv")
chars = msd.NAMES; X = df[chars].to_numpy(float); N = 100; X1, X2 = X[:N], X[N:]
rob = reweighted_fast_mcd(X1, 0.75, 500, 0.975, np.random.default_rng(2026)); cl = classical_estimate(X1)
sd = np.sqrt(np.diag(rob.cov))
print(pd.DataFrame({"published centre": msd.CENTRE, "demo robust centre": rob.center,
                    "published SD": msd.SD, "demo robust SD": sd}, index=chars).round(3))
print("excluded reference days:", df.Date[:N][rob.weights == 0].tolist())
print(f"det(robust)/det(classical) = {np.linalg.det(rob.cov)/np.linalg.det(cl.cov):.2f}")

Phase                 I       II
Blaine    mean  361.420  363.900
          std    17.097   15.840
Residue45 mean   12.544   10.378
          std     1.747    1.057
FreeLime  mean    0.930    0.870
          std     0.114    0.083
LOI       mean    4.401    4.404
          std     0.583    0.600
SO3       mean    1.792    1.917
          std     0.138    0.105
           published centre  demo robust centre  published SD  demo robust SD
Blaine              356.815             361.903        17.872          17.743
Residue45            12.492              12.564         1.800           1.809
FreeLime              0.942               0.936         0.108           0.116
LOI                   4.259               4.323         0.378           0.412
SO3                   1.780               1.803         0.126           0.133
excluded reference days: ['2022-01-03', '2022-01-25', '2022-01-26', '2022-02-25', '2022-02-28', '2022-04-28', '2022-05-13']
det(robust)/det(classical) = 0.44


In [10]:
spec = pd.read_csv(ROOT / "data" / "specifications_demo.csv").set_index("characteristic").reindex(chars)
crit = pd.DataFrame(spec_driven_shifts(chars, rob.center, sd, spec.LSL.to_numpy(float), spec.USL.to_numpy(float), 0.01))
print(crit.round(3).to_string(index=False))
consts = pd.read_csv(ROOT / "data" / "calibration_constants.csv")
lims = {l: float(consts[(consts.chart=="maxm")&(consts.method=="fastmcd")&(consts.N==100)&(consts.p==5)&np.isclose(consts.lam,l)].limit.iloc[0])
        for l in (0.1, 0.3, 0.5, 0.7)}
shifts = []
for _, r in crit.iterrows():
    j = chars.index(r.characteristic)
    if r.critical_mean_shift_sd > 0:
        dv = np.zeros(5); dv[j] = r.direction * r.critical_mean_shift_sd
        shifts.append(Shift(f"{r.characteristic} mean → 1.0% NC", dv, np.ones(5)))
    if r.critical_var_factor > 1:
        v = np.ones(5); v[j] = r.critical_var_factor
        shifts.append(Shift(f"{r.characteristic} spread → 1.0% NC", np.zeros(5), v))
shifts += [uniform_shift(5, 1, 1, "a=1, b=1"), uniform_shift(5, 0, 1.4, "a=0, b=1.4")]
charts = [dict(name=f"λ = {l}", chart="maxm", method="fastmcd", lam=l, limit=v) for l, v in lims.items()]
t0 = time.perf_counter()
study = pd.DataFrame(evaluate_arl(charts, shifts, rob.cov / np.outer(sd, sd), N, 500, 10000, 2027, 0.75, 500, 0.975))
t_study = time.perf_counter() - t0
piv = study.pivot(index="shift", columns="chart", values="ARL")[[f"λ = {l}" for l in lims]]
print(piv.round(1).to_string()); print(f"ARL study: {t_study:.0f} s")
print("fastest lambda per shift:", piv.idxmin(axis=1).to_dict())

characteristic nearer_limit  distance_sd  nonconforming_now  critical_mean_shift_sd  direction  critical_var_factor
        Blaine          LSL        3.207              0.001                   0.881       -1.0                1.900
     Residue45          USL        3.005              0.001                   0.679        1.0                1.669
      FreeLime          USL        3.154              0.001                   0.828        1.0                1.838
           LOI          USL        1.641              0.050                  -0.685        1.0                0.498
           SO3          USL        2.976              0.003                   0.650        1.0                1.637


chart                       λ = 0.1  λ = 0.3  λ = 0.5  λ = 0.7
shift                                                         
Blaine mean → 1.0% NC         178.3    236.9    241.8    241.0
Blaine spread → 1.0% NC        78.2     85.3     88.4     95.8
FreeLime mean → 1.0% NC       199.3    229.7    251.0    258.5
FreeLime spread → 1.0% NC      90.1     98.6    104.6    111.0
Residue45 mean → 1.0% NC      230.5    265.2    285.7    273.0
Residue45 spread → 1.0% NC    115.3    121.5    125.0    131.8
SO3 mean → 1.0% NC            226.4    283.8    293.6    283.9
SO3 spread → 1.0% NC          116.8    131.3    130.5    138.8
a=0, b=1.4                     30.3     36.5     44.5     47.6
a=1, b=1                       10.3     11.9     14.4     15.5
ARL study: 22 s
fastest lambda per shift: {'Blaine mean → 1.0% NC': 'λ = 0.1', 'Blaine spread → 1.0% NC': 'λ = 0.1', 'FreeLime mean → 1.0% NC': 'λ = 0.1', 'FreeLime spread → 1.0% NC': 'λ = 0.1', 'Residue45 mean → 1.0% NC': 'λ = 0.1', 'Residue45

In [11]:
t0 = time.perf_counter()
path = ermm_path(X2, X1[-1], rob.center, rob.cov, 0.1)
t_mon = time.perf_counter() - t0
tbl = signal_table(df.Date[N:], path["stat"], lims[0.1], 3); k1 = first_signal(tbl)
print(f"signals {tbl.signal.sum()} of {len(tbl)}, first {tbl.date.iloc[k1].date()}, after a stop {(tbl.signal & tbl.after_stop).sum()}")
comp = component_summary(path["U"], path["V"], k1); print({k: round(v, 3) for k, v in comp.items()})
dev = characteristic_deviations(X2, chars, rob.center, rob.cov, k1); print(dev.round(3).to_string(index=False))
ocap = pd.read_csv(ROOT / "data" / "ocap_finish_mill.csv")
print(rank_actions(dev, ocap, 0.5, 20)[["characteristic", "change", "process_unit"]].to_string(index=False))
props = ["InitialSetting_min", "FinalSetting_min", "Strength3d_kgcm2", "Strength7d_kgcm2"]
print(property_comparison(df[props], np.r_[np.ones(N, bool), np.zeros(60, bool)],
                          np.r_[np.zeros(N + k1, bool), np.ones(60 - k1, bool)]).round(3).to_string(index=False))

signals 6 of 60, first 2022-10-04, after a stop 0
{'days': 40, 'mean_abs_U': 0.638, 'mean_abs_V': 1.221, 'in_control_mean_abs': 0.798, 'share_location_dominant': 0.2, 'share_V_negative': 0.675}
characteristic  reference_centre  mean_after  shift_robust_sd  reference_sd  sd_after  sd_ratio  sd_change_pct
        Blaine           361.903     363.600            0.096        17.743    14.604     0.823        -17.693
     Residue45            12.564      10.567           -1.104         1.809     1.031     0.570        -43.023
      FreeLime             0.936       0.890           -0.396         0.116     0.065     0.563        -43.730
           LOI             4.323       4.349            0.062         0.412     0.602     1.461         46.129
           SO3             1.803       1.878            0.564         0.133     0.076     0.572        -42.825
characteristic                     change               process_unit
     Residue45 level down, spread smaller    Separator and mill load
  

## 7 Computing times (single processor core)

In [12]:
reps = 2000; t0 = time.perf_counter()
for _ in range(reps): ermm_path(X2[:1], X2[-1], rob.center, rob.cov, 0.1)
t_day = (time.perf_counter() - t0) / reps
cal = json.loads(calib_cache.CACHE.read_text())
times = pd.DataFrame([dict(chart=name[(v["chart"], v["method"])], lam=v["lam"], minutes=v["seconds"]/60) for v in cal.values()])
print(f"Fast-MCD estimate: {1000*t_mcd:.0f} ms; daily chart update: {1e3*t_day:.2f} ms; Phase II path of 60 days: {1e3*t_mon:.1f} ms")
print(times.groupby("chart").minutes.agg(["min", "max"]).round(2))

Fast-MCD estimate: 12 ms; daily chart update: 0.32 ms; Phase II path of 60 days: 1.4 ms
                   min   max
chart                       
EMM               0.24  0.29
ERMM              2.30  2.35
MEWMA, Fast-MCD   2.38  2.38
MEWMA, classical  0.06  0.06
